In [10]:
import os
import uuid
import pickle
import sqlite3
from datetime import datetime
import gradio as gr
from langchain.llms import Ollama

# SETUP
llm = Ollama(model="qwen3:4b")
db_path = "chatbot_memory.db"

def init_db():
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS chat_memory (
            id TEXT PRIMARY KEY,
            user_id TEXT,
            summary TEXT,
            chat_blob BLOB,
            timestamp TEXT
        )
    ''')
    conn.commit()
    conn.close()

init_db()

def generate_user_id():
    return str(uuid.uuid4())

def create_summary(chat_history):
    if not chat_history:
        return "Không có nội dung"
    u, b = chat_history[-1]
    return f"{u[:20]} → {b[:20]}"

def chat(user_input, history, user_id):
    prompt = user_input
    response = llm(prompt)
    history.append((user_input, response))
    return history, history

def save_to_sqlite(user_id, summary, chat_history):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    chat_blob = pickle.dumps(chat_history)
    cursor.execute('''
        INSERT INTO chat_memory (id, user_id, summary, chat_blob, timestamp)
        VALUES (?, ?, ?, ?, ?)
    ''', (
        user_id + "_" + str(uuid.uuid4()),
        user_id,
        summary,
        chat_blob,
        datetime.now().isoformat()
    ))
    conn.commit()
    conn.close()

def save_hybrid_memory(user_id, chat_history):
    if not chat_history:
        return [], update_saved_chats(), "Không có nội dung để lưu."
    summary = create_summary(chat_history)
    save_to_sqlite(user_id, summary, chat_history.copy())
    return [], update_saved_chats(), f"Đã lưu: {summary}"

def update_saved_chats():
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("SELECT summary FROM chat_memory ORDER BY timestamp DESC")
    results = [row[0] for row in cursor.fetchall()]
    conn.close()
    return results

def load_conversation(selected_summary):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("SELECT chat_blob FROM chat_memory WHERE summary = ?", (selected_summary,))
    row = cursor.fetchone()
    conn.close()
    return pickle.loads(row[0]) if row else []

def semantic_search(keyword):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("SELECT chat_blob FROM chat_memory WHERE summary LIKE ?", (f"%{keyword}%",))
    row = cursor.fetchone()
    conn.close()
    return pickle.loads(row[0]) if row else []

# GIAO DIỆN
with gr.Blocks() as demo:
    gr.Markdown("Chatbot SQLite")

    with gr.Row():
        with gr.Column(scale=1):
            saved_chats = gr.Radio(label="Đoạn chat đã lưu", choices=[], interactive=True)
            search_box = gr.Textbox(placeholder="Từ khóa gợi nhớ...")
            search_btn = gr.Button("Gợi nhớ")

        with gr.Column(scale=4):
            chatbot = gr.Chatbot()
            msg = gr.Textbox(placeholder="Nhập câu hỏi...")
            user_id = gr.Textbox(value=generate_user_id(), visible=False)
            state = gr.State([])

            with gr.Row():
                send = gr.Button("Gửi")
                save_btn = gr.Button("Lưu & Tạo mới")
                status = gr.Textbox(label="Trạng thái", interactive=False)

    send.click(chat, [msg, state, user_id], [chatbot, state])
    save_btn.click(save_hybrid_memory, [user_id, state], [state, saved_chats, status])
    saved_chats.change(load_conversation, saved_chats, chatbot)
    search_btn.click(semantic_search, [search_box], chatbot)

demo.launch()


ImportError: cannot import name 'get_token' from 'huggingface_hub' (c:\Users\DELL\anaconda3\Lib\site-packages\huggingface_hub\__init__.py)

In [4]:
!pip install huggingface_hub==0.30.1
!pip install transformers==4.41.2
!pip install sentence-transformers==2.6.1


In [6]:
!pip install --upgrade huggingface_hub


  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 0.30.1
    Uninstalling huggingface-hub-0.30.1:
      Successfully uninstalled huggingface-hub-0.30.1


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2025.7.0 which is incompatible.


In [8]:
import huggingface_hub
print(huggingface_hub.__version__)


0.17.3


In [9]:
!pip install huggingface_hub==0.14.1


  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 0.33.5
    Uninstalling huggingface-hub-0.33.5:
      Successfully uninstalled huggingface-hub-0.33.5


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
accelerate 1.9.0 requires huggingface_hub>=0.21.0, but you have huggingface-hub 0.14.1 which is incompatible.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2025.7.0 which is incompatible.
datasets 4.0.0 requires huggingface-hub>=0.24.0, but you have huggingface-hub 0.14.1 which is incompatible.
gradio 5.29.0 requires huggingface-hub>=0.28.1, but you have huggingface-hub 0.14.1 which is incompatible.
gradio-client 1.10.0 requires huggingface-hub>=0.19.3, but you have huggingface-hub 0.14.1 which is incompatible.
sentence-transformers 2.6.1 requires huggingface-hub>=0.15.1, but you have huggingface-hub 0.14.1 which is incompatible.
tokenizers 0.19.1 requires huggingface-hub<1.0,>=0.16.4, but you have huggingface-hub 0.14.1 which is incompatible.
transformers 4.41.2 requires huggingfa

In [11]:
import huggingface_hub
print(huggingface_hub.__version__)  # phải là 0.14.1


0.17.3


In [ ]:
pip uninstall huggingface_hub -y
pip install huggingface_hub==0.14.1